# Real-data capstone: history matching against measurements

_Optional. Where decision-support reactive transport modelling actually lives._


Everything in [part1_03](../part1_03_obs_weights_and_truth/dizon_obs_weights_and_truth.ipynb) through [part1_08](../part1_08_optimization/dizon_optimization.ipynb) was run against a **synthetic truth** (see [`CONTEXT.md`](../../CONTEXT.md)) - a single realisation we lifted out of the prior ensemble and deliberately chose so the decision could go either way. That choice bought us something precious: we always knew the answer. Every posterior we drew could be scored against a truth that, by construction, lived inside the prior. "Did we cover the truth?" was a question with a yes/no answer.

This notebook throws that luxury away. We swap the synthetic truth for the **measured data** - the real DIZON field observations from Someren - and run the same machinery. Mechanically, very little changes: a handful of `obsval`s and a noise model. Conceptually, almost everything changes. The misfit we minimise is no longer honest evidence about parameters; it is parameters *plus* model-structural error, tangled together with no way to separate them. And the question that organised the whole series - "did we cover the truth?" - becomes literally unanswerable at the place we care about most.


This is the honest end of the series. Synthetic experiments teach the methods cleanly because they remove the one thing real work never removes: the model is wrong, and you cannot measure by how much. We keep this as a capstone, and as opt-in, on purpose - structural error is a lesson of its own, not noise to scatter through the middle of the others.

Because this notebook is a **skeleton**, the code cells below are commented stubs. They name the calls and the machinery they reuse from [part1_03](../part1_03_obs_weights_and_truth/dizon_obs_weights_and_truth.ipynb) and [part1_05](../part1_05_dsi_basics/dizon_dsi_basics.ipynb), but they are not wired to run until the upstream notebooks and the `prebaked/` artifacts are in place.


### Admin

We reuse the trained DSI emulator and the prior Monte Carlo obs ensemble from the core sequence - nothing here asks for a fresh run of the full DIZON model (each one costs ~6 min on a laptop, and the capstone changes only the *data*, not the model). The new ingredient is `data/obs_chem_cleaned.csv`: the cleaned field measurements, the same file part1_01 used to sanity-check the freshly built model.

Imports first, then the prerequisite check pointing back upstream.


In [ ]:
import os
import shutil
import warnings
warnings.filterwarnings("ignore")
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import psutil

import sys
import pyemu
import flopy
assert "dependencies" in flopy.__file__
assert "dependencies" in pyemu.__file__
sys.path.insert(0, "..")
import herebedragons as hbd

This notebook sits at the end of the part1 sequence. It needs the DSI emulator trained in [part1_05](../part1_05_dsi_basics/dizon_dsi_basics.ipynb) and the obs setup (species, weights, obsid scheme) from [part1_03](../part1_03_obs_weights_and_truth/dizon_obs_weights_and_truth.ipynb). Check those exist before going further:


In [ ]:
# TODO: prerequisite check - point at the upstream notebooks if their outputs are missing.
#
# # the trained DSI emulator from part1_05
# dsi_t_d = Path(os.path.join('..', 'part1_05_dsi_basics', 'dsi_template'))
# if not dsi_t_d.exists():
#     raise Exception("you need to run the '../part1_05_dsi_basics/dizon_dsi_basics.ipynb' notebook")
#
# # the prior Monte Carlo source - resolved in the canonical order the series uses:
# #   1. the tracked, thinned prebaked obs ensemble (../../prebaked/prior_mc_obs_ensemble.jcb)
# #   2. a full prior MC run into the repo-root master dir (../../master_priormc)
# prebaked_oe = Path('..') / '..' / 'prebaked' / 'prior_mc_obs_ensemble.jcb'
# pmc_d = Path('..') / '..' / 'master_priormc'
# if not (prebaked_oe.exists() or (pmc_d / 'pest.pst').exists()):
#     raise Exception("you need to run the '../part1_04_prior_mc/dizon_prior_mc.ipynb' notebook "
#                     "(prior Monte Carlo results not found)")

We work in a fresh template inside *this* notebook's directory, copied from the part1_05 DSI template so we never disturb the synthetic-truth version. The whole capstone is "the same emulator, different data".


In [ ]:
# TODO: copy the trained DSI template into a capstone working dir (this notebook's folder).
#
# t_d = Path('dsi_template_realdata')
# if t_d.exists():
#     shutil.rmtree(t_d)
# shutil.copytree(dsi_t_d, t_d)
#
# dpst = pyemu.Pst(str(t_d / 'dsi.pst'))

## What changes mechanically

Start with the easy part: the plumbing. In the core sequence, the observation values we conditioned on came from one prior realisation - the synthetic truth. To use measured data instead, we change exactly two things:

1. **The observation values** (`obsval`) become the field measurements.
2. **The noise model** changes, because measurement reality is messier than the clean species-specific noise we *assigned* in part1_03.

Weights stay zeroed after day 252 (the [history period](../../CONTEXT.md) ends at the [decision date](../../CONTEXT.md), exactly as before), the conditioning species are the same (SO₄, O₂, NO₃, pH, Tmp), and the held-back cations are still held back. The forecast is still peak SO₄ at the supply well (`welopt`) over the [supply period](../../CONTEXT.md). The *workflow* is unchanged. Only the numbers feeding into it move.

### The swap is already wired into the build

Conveniently, the model build emits a single file that aligns simulated outputs with measurements row-for-row: `_obs.conc.simvsmeas.csv`. It carries columns `time, obsid, layer, variable` plus a `sim` column (the model's output) and a `meas` column (the field value, or `1e30` where no measurement exists). part1_03 used the `sim` side of this file to pull the synthetic truth; the capstone uses the `meas` side.

Load it and see how sparse real data actually is - the `meas` column is mostly the `1e30` no-data flag:


In [ ]:
# TODO: load the sim-vs-meas alignment file (written by the model build, lives in the prior-MC workspace).
#        Use the same prior-MC source resolved in the prereq cell (the prebaked
#        prior_mc_obs_ensemble.jcb, falling back to the repo-root master_priormc).
#
# meas = pd.read_csv(pmc_d / '_obs.conc.simvsmeas.csv')
# meas['layer'] = meas['layer'].astype(int)
# meas['variable'] = meas['variable'].astype(str).str.lower()
# meas = meas.sort_values(['obsid', 'variable', 'layer', 'time'])
#
# # real measurements are flagged: meas < 1e30. Everything else is a placeholder.
# n_real = (meas['meas'] < 1e30).sum()
# print(f'{n_real} real measurements out of {len(meas)} aligned rows')   # ~4,100 of ~222,000

Now overwrite the observation values. This is the whole mechanical swap: where part1_03 set `obsval` from the chosen truth realisation, we merge in the `meas` column on `(obsid, variable, time)`. This is exactly the `USE_MEASURED_DATA` switch the prototype carried - flipped on.


In [ ]:
# TODO: replace synthetic obsvals with measured values (the real-data switch).
#
# dobs = dpst.observation_data
# dobs['weight'] = 0.0
#
# _ = dobs.copy()
# _['time'] = _['time'].astype(float)
# obsvals = (_.merge(meas, on=['obsid', 'variable', 'time'], how='left', suffixes=('', '_meas'))
#             .set_index('obsnme')['meas'])
# dobs.loc[obsvals.index, 'obsval'] = obsvals.values
#
# # rows that got no measurement keep the no-data flag and will never be weighted
# has_meas = dobs['obsval'] < 1e30

### Noise from measurement realities

In part1_03 the noise was a *modelling assumption* we narrated honestly: proportional-with-a-floor for concentrations, absolute for pH (~0.1) and temperature (~0.5 °C). With synthetic data that assumption was self-consistent - we added exactly that noise to a clean realisation, so the noise model was, by construction, correct.

Field data does not work that way. The same instrument-and-handling story still applies (detection limits, rounding, sampling and analytical scatter), so the *form* of the noise model carries over. But now it is a genuine guess about errors we cannot audit, and - this is the uncomfortable part - it is also the only knob we have to absorb model-structural error. More on that below. For the mechanics, we reuse the species-specific noise table from part1_03 unchanged:


In [ ]:
# TODO: assign weights/noise on the measured, history-period, conditioning-species obs only.
#        Reuse the species-specific noise model from part1_03 (cell with PROP_FRAC/PROP_FLOOR/ABS_SIGMA):
#        proportional-with-a-floor for concentrations, absolute for pH (~0.1) and Tmp (~0.5 C).
#        pe is never conditioned on.
#
# nz = dobs.loc[(dobs['time'].astype(float) <= 252) & has_meas].copy()
# # nz = nz.loc[nz['variable'].isin(['so4', 'o0', 'no3', 'ph', 'tmp'])]   # conditioning species
#
# # the part1_03 species-specific noise model, inline:
# PROP_FRAC  = 0.07                                        # proportional sigma for concentrations
# PROP_FLOOR = {'so4': 5.0e-5, 'o0': 2.0e-5, 'no3': 2.0e-5}  # absolute floor on the conc sigma (mol/L)
# ABS_SIGMA  = {'ph': 0.1, 'tmp': 0.5}                     # absolute sigma (pH units; deg C)
#
# def obs_sigma(row):
#     if row.variable in ABS_SIGMA:
#         return ABS_SIGMA[row.variable]
#     return max(abs(row.obsval) * PROP_FRAC, PROP_FLOOR.get(row.variable, 0.0))
#
# dobs.loc[nz.index, 'standard_deviation'] = nz.apply(obs_sigma, axis=1)
# dobs.loc[nz.index, 'weight'] = 1.0 / dobs.loc[nz.index, 'standard_deviation']
# dobs.loc[dobs['obsval'] == 0.0, 'weight'] = 0.0   # below-detection zeros carry no information

The phi-factor file (per site:species group) and the noise ensemble are generated exactly as in part1_05 - the emulator does not know or care whether the targets came from a realisation or a field campaign. We rebuild the noise ensemble so it is consistent with the new standard deviations, then point the control file at it:


In [ ]:
# TODO: regenerate phi-factor file and noise ensemble for the new obsvals (part1_05 machinery).
#
# dobs['obgnme'] = dobs.apply(lambda x: f"{x.obsid}:{x.variable}", axis=1)
# phi_factors = pd.Series(index=dobs.obgnme.unique(), dtype=float)
# phi_factors.loc[:] = 1.0 / dobs.obgnme.nunique()
# phi_factors.to_csv(os.path.join(t_d, 'ies_phi_factors.csv'), header=False)
# dpst.pestpp_options['ies_phi_factor_file'] = 'ies_phi_factors.csv'
#
# nreals = dpst.pestpp_options['ies_num_reals']
# noise = pyemu.ObservationEnsemble.from_gaussian_draw(dpst, num_reals=nreals)
# # ... fill noise per obgnme from standard_deviation, zero out below-detection obs ...
# noise.to_binary(os.path.join(t_d, 'noise.jcb'))
# dpst.pestpp_options['ies_observation_ensemble'] = 'noise.jcb'

And that is the entire mechanical difference. Two columns and a noise file. If the only goal were "run the workflow on real data", we would be done. The rest of this notebook is about why being done here would be dishonest.


## What breaks conceptually

Three things break, and they are the three things that made the synthetic series legible.


### 1. The misfit is now ambiguous

Against the synthetic truth, residuals were honest. The truth came from the same model with the same structure; any misfit was either a parameter we had not yet found, or the noise we ourselves added. Drive phi down and you were unambiguously moving toward the parameters that generated the data.

Against measured data, a residual is the sum of (at least) two terms we cannot separate:

$$ r = \underbrace{d_{\text{obs}} - M(\boldsymbol{\theta})}_{\text{what we minimise}} = \underbrace{(d_{\text{obs}} - M_{\text{true}}(\boldsymbol{\theta}^*))}_{\text{measurement noise}} \;+\; \underbrace{(M_{\text{true}}(\boldsymbol{\theta}^*) - M(\boldsymbol{\theta}))}_{\text{structural error + parameter error}} $$

The DIZON model is a faithful recast of the field chemistry, not a reproduction of it (854 days abstracted to 728; a simplified redox network; a regular grid standing in for real heterogeneity). $M$ is not $M_{\text{true}}$. So when the ensemble smoother lowers phi, it cannot tell whether it found better parameters or merely contorted the parameters to paper over a structural gap. The same noise model that was a clean assumption against synthetic data now silently doubles as the budget for structural error - widen it and the model "fits"; tighten it and it cannot. Neither choice is provably right.


The practical symptom: prior-data conflict you cannot resolve by sampling. Some measured series will sit outside the prior envelope no matter which parameters you draw - not because the prior is too narrow, but because no setting of K, porosity, pyrite abundance and pyrite rate can make *this* model reproduce *that* observation. Run the prior-vs-measured overlay (part1_04's 1-to-1 / timeseries machinery, measured data on top) and read it as a diagnostic of model adequacy, not of parameter values:


In [ ]:
# TODO: overlay prior ensemble against measured data (part1_04 plotting machinery).
#        Look for measured series that fall OUTSIDE the prior envelope -> structural error,
#        not something history matching can fix.
#
# oe_prior = pyemu.ObservationEnsemble.from_binary(
#     pst=dpst, filename=os.path.join(t_d, 'dsi.0.obs.jcb'))
# for obgnme in dobs.loc[dobs.weight > 0].obgnme.unique():
#     # plot prior ribbon (grey) + measured points (red); flag any measured point outside the ribbon
#     ...

### 2. Truth coverage is unknowable - and worst exactly where it matters

The series' recurring scorecard was "does the posterior bracket the truth?". That question needs a truth. With measured data there isn't one; there are only more measurements, themselves uncertain. We can still ask "does the posterior bracket the *held-out measurements*?" - a useful validation - but it is a weaker claim, because those held-out points carry their own noise and their own structural mismatch.

And here the DIZON case delivers its sharpest lesson. **The supply well does not exist in the field data.** The forecast - peak SO₄ at the supply well (`welopt`) during the supply period - is a counterfactual: a well that was never drilled, pumping water that was never produced. The measured campaign covers monitoring sites only (`wp*`, `pp1`, `ip2`). There are zero field measurements at the forecast location, ever:

In [ ]:
# TODO: confirm the forecast location is unobserved in the field.
#
# welopt_rows = meas[meas['obsid'].astype(str).str.startswith('welopt')]
# print('measured values at the supply well:', (welopt_rows['meas'] < 1e30).sum())   # -> 0
#
# # contrast: monitoring sites that DO carry measurements
# observed_sites = sorted(meas.loc[meas['meas'] < 1e30, 'obsid'].astype(str).unique())
# print('sites with real data:', observed_sites)   # wp*, pp1, ip2 - never welopt

This is not a quirk of one dataset; it is the normal condition of decision support. The forecast almost always lives somewhere - or somewhen - you have no data: a future stress, a location you cannot monitor, a quantity no instrument measures. The synthetic truth let us *pretend* we could check the forecast directly. Real data removes the pretence. The posterior forecast distribution is the answer, and there is no line on the plot to check it against. All you can validate is whether the model reproduces the data you *do* have, and then argue - on physical grounds, not statistical ones - that this earns trust in an extrapolation it can never confirm.


### 3. The decision still has to be made

None of this is an excuse to stop. The decision question is unchanged: how much sulfate will the supplied water carry, and how sure are we? Treatment capacity gets designed to the P95 of peak SO₄, so the answer is a *distribution*. Run the conditioning on the emulator exactly as in part1_05 (runstor; `ies_multimodal_alpha=0.99`; the noise ensemble built above), then read the prior-vs-posterior forecast — median and P95 — off the ensemble. The plot looks identical to the synthetic one - minus the truth line, plus a much larger asterisk about what "posterior" now means.

In [ ]:
# TODO: condition the DSI emulator on the measured data (part1_05 runstor workflow) and read the risk.
#
# dpst.pestpp_options['ies_multimodal_alpha'] = 0.99   # see the part1_05 explanation cell
# dpst.control_data.noptmax = 3
# dpst.write(os.path.join(t_d, 'dsi.pst'), version=2)
# hbd.get_bins(t_d)
#
# num_workers = psutil.cpu_count(logical=False)
# m_d = Path('master_capstone')
# pyemu.os_utils.start_workers(t_d, 'pestpp-ies', 'dsi.pst',
#                              num_workers=num_workers, worker_root='.', master_dir=str(m_d))

In [ ]:
# TODO: prior vs posterior forecast distribution - NO truth line this time.
#
# _pst = pyemu.Pst(str(m_d / 'dsi.pst'))
# obsen = _pst.ies.obsen
# prior_fc = obsen.loc[0, _pst.forecast_names]
# post_fc  = obsen.loc[_pst.control_data.noptmax, _pst.forecast_names]
#
# # plot both histograms; report median and P95 of peak SO4 prior vs posterior -
# # the P95 is the number treatment capacity gets designed to.
# # Optional risk lens (a contractual trigger, not a drinking-water limit):
# # TRIGGER = 90.0   # mg/L
# # p_prior = (prior_fc > TRIGGER).mean(); p_post = (post_fc > TRIGGER).mean()

Optionally, the fidelity check still runs (it never needed the truth - it compares the emulator against the full model on held-out realisations, both synthetic). It tells you the emulator is faithful to the *model*. It says nothing about whether the model is faithful to the *aquifer*. Keep the two questions separate; conflating them is the most common way real-data analyses fool themselves.


## What the synthetic/real contrast taught

Running both ends of the series back to back is the point of the capstone. The contrast is the lesson:

- **The synthetic truth was a teaching instrument, not a crutch.** It let every method show its payoff against a known answer, so you could see *that the method works* before trusting it where the answer is hidden. That is the only honest order to learn in.
- **The mechanics are trivial; the epistemics are everything.** Swapping to real data took two columns and a noise file. What changed was not the code but what the numbers coming out of it are allowed to claim.
- **Misfit stops being evidence about parameters.** With a wrong model - and every model is wrong - low phi can mean good parameters or well-disguised structural error, and the data cannot tell you which. The noise model quietly becomes your structural-error budget, and you are choosing it, not measuring it.
- **The forecast lives where you have no data.** The supply well was never drilled. Decision support is, almost by definition, extrapolation to an unobserved place or time; "did we cover the truth?" is a question you only get to answer in synthetic worlds.
- **You still have to commit.** The decision date does not wait for certainty. The job is not to remove the ambiguity - you can't - but to carry it honestly into the decision, state the structural assumptions the forecast rests on, and be clear about what the data can and cannot underwrite.


That last point is the whole reason the series chose emulation and uncertainty quantification over calibrate-then-forget. A single calibrated model hands the decision-maker a number and hides the ambiguity. The workflow you have now hands them a forecast distribution — median and P95 — a stated noise model, and an explicit structural caveat. It does not make the model right. It makes the modelling honest - and with reactive transport run costs of ~6 min per realisation, the emulator is what makes that honesty affordable. That is where real decision-support reactive transport modelling lives, and it is where the series ends.


### Where to go next

- Revisit [part1_07 dataworth](../part1_07_dataworth/dizon_dataworth.ipynb): with real data, the held-back cations question becomes "is it worth the field cost of sampling them?" - a real budget decision, not a thought experiment. Note that the field campaign only ever measured some cations (Ca, Na, Fe) and never others (Mg, K), which is itself a dataworth result the world handed you.
- Carry the structural-error framing into [part1_08 optimization](../part1_08_optimization/dizon_optimization.ipynb): a Pareto front read off a structurally-uncertain posterior needs its caveats stated on the figure, not buried in a footnote.
- For the deeper treatment of structural error and model adequacy, see the GMDSI publications linked from the [series README](../../README.md).
